<a href="https://colab.research.google.com/github/jluisro-byte/Agent-Skills-for-Context-Engineering/blob/main/Resistor_VLM_01_YOLO_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Celda 1 — Título y propósito
# Resistor-VLM — YOLO11 Baseline

Pipeline de detección de resistencias en diagramas eléctricos.

## Objetivo

Entrenar y evaluar un detector YOLO11n capaz de localizar resistencias
en diagramas de circuitos eléctricos.

## Dataset

- 266 imágenes
- 1,350 resistencias anotadas
- 132 fuentes originales
- Split por `final_source_id` para evitar data leakage

| Split | Imágenes | Objetos |
|---|---:|---:|
| Train | 186 | 945 |
| Validation | 40 | 203 |
| Test | 40 | 202 |

El conjunto TEST permanece separado del entrenamiento y de la selección
del modelo.

SyntaxError: invalid syntax (3206492920.py, line 4)

In [6]:
# Celda "2"
#Montar Drive
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

print()
print("Drive mounted:", DRIVE_ROOT.exists())
print("Drive root:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Drive mounted: True
Drive root: /content/drive/MyDrive


In [7]:
# Celda 3 Instalar Dependiencias
!pip install -q ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.2 MB/s eta 0:00:00


In [8]:
# celda 4 Imports y configuración global
from pathlib import Path
import yaml
import torch
import time

from ultralytics import YOLO

PROJECT_ROOT = Path("/content/drive/MyDrive/Resistor-VLM")

DATASET_ROOT = PROJECT_ROOT / "datasets" / "processed"
DATASET_YAML = DATASET_ROOT / "dataset.yaml"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"

TRAINING_ROOT = OUTPUT_ROOT / "training"
EVALUATION_ROOT = OUTPUT_ROOT / "evaluation"

MODEL_NAME = "yolo11n.pt"

IMAGE_SIZE = 416
BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_ROOT)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Project: /content/drive/MyDrive/Resistor-VLM
Dataset: /content/drive/MyDrive/Resistor-VLM/datasets/processed
PyTorch: 2.11.0+cpu
Device: cpu


In [9]:
# Celda 5 — Crear YAML portable
dataset_config = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "resistor"
    }
}

with open(DATASET_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        dataset_config,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print(DATASET_YAML.read_text())

path: /content/drive/MyDrive/Resistor-VLM/datasets/processed
train: images/train
val: images/val
test: images/test
names:
  0: resistor



In [10]:
# Celda 6 — Preflight del dataset
expected = {
    "train": 186,
    "val": 40,
    "test": 40,
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("=" * 60)
print("DATASET PREFLIGHT")
print("=" * 60)

assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"

total_images = 0
total_labels = 0

for split in ["train", "val", "test"]:

    image_dir = DATASET_ROOT / "images" / split
    label_dir = DATASET_ROOT / "labels" / split

    assert image_dir.exists(), f"Missing: {image_dir}"
    assert label_dir.exists(), f"Missing: {label_dir}"

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_dir.glob("*.txt"))

    print(
        f"{split.upper():5} | "
        f"images={len(images):3} | "
        f"labels={len(labels):3}"
    )

    assert len(images) == expected[split]
    assert len(labels) == expected[split]

    total_images += len(images)
    total_labels += len(labels)

assert total_images == 266
assert total_labels == 266

print("-" * 60)
print("Total images:", total_images)
print("Total labels:", total_labels)
print("Dataset structure: OK")

DATASET PREFLIGHT
TRAIN | images=186 | labels=186
VAL   | images= 40 | labels= 40
TEST  | images= 40 | labels= 40
------------------------------------------------------------
Total images: 266
Total labels: 266
Dataset structure: OK


In [11]:
#Celda 7 — Verificar anotaciones
expected_objects = {
    "train": 945,
    "val": 203,
    "test": 202,
}

expected_negatives = {
    "train": 6,
    "val": 2,
    "test": 2,
}

print("=" * 60)
print("ANNOTATION CHECK")
print("=" * 60)

total_objects = 0

for split in ["train", "val", "test"]:

    label_dir = DATASET_ROOT / "labels" / split

    objects = 0
    negatives = 0

    for label_file in label_dir.glob("*.txt"):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            negatives += 1

        objects += len(lines)

    print(
        f"{split.upper():5} | "
        f"objects={objects:3} | "
        f"negatives={negatives}"
    )

    assert objects == expected_objects[split]
    assert negatives == expected_negatives[split]

    total_objects += objects

assert total_objects == 1350

print()
print("Total objects:", total_objects)
print("Annotation counts: OK")

ANNOTATION CHECK
TRAIN | objects=945 | negatives=6
VAL   | objects=203 | negatives=2
TEST  | objects=202 | negatives=2

Total objects: 1350
Annotation counts: OK


In [ ]:
#Celda 8 — Entrenamiento baseline
# Ojo solo se dede de ejecutar una sola vez el entrenamiento
#Esto crea nuestro baseline reproducible.
#No se necesita ejecutarlo ahora, porque ya tenemos exp002.

model = YOLO(MODEL_NAME)

start = time.time()

results = model.train(
    data=str(DATASET_YAML),

    epochs=50,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,

    device=DEVICE,
    workers=2,

    project=str(TRAINING_ROOT),
    name="baseline_yolo11n_50ep",

    seed=42,
    deterministic=True,

    pretrained=True,
    verbose=True
)

elapsed = time.time() - start

print("=" * 60)
print("TRAINING FINISHED")
print("=" * 60)

print(f"Training time: {elapsed / 60:.2f} minutes")

In [12]:
# Celda 9 Seleccionar Modelo Existente
BEST_MODEL = (
    PROJECT_ROOT
    / "outputs"
    / "training"
    / "exp002_yolo11n_cpu_50ep"
    / "weights"
    / "best.pt"
)

assert BEST_MODEL.exists(), f"Model not found: {BEST_MODEL}"

print("Model:")
print(BEST_MODEL)

Model:
/content/drive/MyDrive/Resistor-VLM/outputs/training/exp002_yolo11n_cpu_50ep/weights/best.pt


In [13]:
# Evaluacion Final TEST
model = YOLO(str(BEST_MODEL))

start = time.time()

metrics = model.val(
    data=str(DATASET_YAML),

    split="test",

    device=DEVICE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=2,

    project=str(EVALUATION_ROOT),
    name="baseline_test",

    verbose=True
)

elapsed = time.time() - start

Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.2 ms, read: 0.1±0.0 MB/s, size: 15.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Resistor-VLM/datasets/processed/labels/test.cache... 40 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40 4.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.1s/it 30.8s
                   all         40        202      0.821      0.771       0.82      0.373
Speed: 1.6ms preprocess, 74.0ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/drive/MyDrive/Resistor-VLM/outputs/evaluation/baseline_test


In [14]:
# Celda 11  Resultados
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map5095 = float(metrics.box.map)

print("=" * 60)
print("RESISTOR-VLM — BASELINE TEST RESULTS")
print("=" * 60)

print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"mAP50:      {map50:.4f}")
print(f"mAP50-95:   {map5095:.4f}")

print()
print(f"Evaluation time: {elapsed / 60:.2f} min")

print("=" * 60)

RESISTOR-VLM — BASELINE TEST RESULTS
Precision:  0.8208
Recall:     0.7711
mAP50:      0.8200
mAP50-95:   0.3735

Evaluation time: 0.67 min


In [15]:
# Celda 12 Registrar el Experimiento
#cuando tengamos EXP003, EXP004, YOLO11s, OCR, etc.,
# necesitaremos una historia experimental.
import pandas as pd

experiment = pd.DataFrame([{
    "experiment": "baseline_yolo11n_50ep",
    "model": "YOLO11n",
    "epochs": 50,
    "imgsz": IMAGE_SIZE,
    "train_images": 186,
    "val_images": 40,
    "test_images": 40,
    "test_objects": 202,
    "precision": precision,
    "recall": recall,
    "mAP50": map50,
    "mAP50_95": map5095,
    "device": DEVICE,
    "seed": 42,
}])

experiment

,experiment,model,epochs,imgsz,train_images,val_images,test_images,test_objects,precision,recall,mAP50,mAP50_95,device,seed
0,baseline_yolo11n_50ep,YOLO11n,50,416,186,40,40,202,0.820823,0.771079,0.819981,0.373459,cpu,42


In [16]:
# Celda 13 guarda el experimiento
EXPERIMENTS_FILE = OUTPUT_ROOT / "experiments.csv"

if EXPERIMENTS_FILE.exists():

    previous = pd.read_csv(EXPERIMENTS_FILE)

    previous = previous[
        previous["experiment"] != "baseline_yolo11n_50ep"
    ]

    experiments = pd.concat(
        [previous, experiment],
        ignore_index=True
    )

else:
    experiments = experiment

experiments.to_csv(
    EXPERIMENTS_FILE,
    index=False
)

print("Saved:")
print(EXPERIMENTS_FILE)

experiments

Saved:
/content/drive/MyDrive/Resistor-VLM/outputs/experiments.csv


,experiment,model,epochs,imgsz,train_images,val_images,test_images,test_objects,precision,recall,mAP50,mAP50_95,device,seed
0,baseline_yolo11n_50ep,YOLO11n,50,416,186,40,40,202,0.820823,0.771079,0.819981,0.373459,cpu,42
